In [21]:
from passwords import *
from pprint import pprint
import pandas as pd
try:
    import pyodbc
except ModuleNotFoundError:
    ! pip install pyodbc
    import pyodbc
import os
try:
    import boto3
except ModuleNotFoundError:
    ! pip install boto3
    import boto3
import datetime as dt
from tqdm import tqdm

#### constants

In [6]:
# project
str_project = os.getcwd().split('\\')[4].replace('_', '-')
print(f'Project: {str_project}')
 # task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'sub task: {str_subtask}')

# dir output
str_dirname_output = './output'
print(str_dirname_output)

Project: 20240509-christian-internship
Task: 03_python
sub task: 01_jupyter_practice
./output


#### push to s3 function

In [22]:
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init boto 3 client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    
    cls_client.upload_file(
        str_local_path,
        str_bucket_name,
        str_bucket_key
    )

#### read in sql query

In [8]:
str_query = open('./sql/SQLQuery2.sql', 'r').read()
pprint(str_query)

('\n'
 'SELECT TSP.bigAccountId, \n'
 'TSP.AmtFinanced, \n'
 'ACC.fltNetChgOff\n'
 '\n'
 'FROM electra.riskdb.analytics.tbltempstaticpool as TSP\n'
 '\n'
 'INNER JOIN riskdb.accountingReports.tblAccounting_LoanCOandNA_ME as ACC on '
 'TSP.bigAccountId = ACC.bigAccountId')


### Read SQL

In [9]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

df = pd.read_sql_query(
    str_query,
    con=conn,
)

# close connection
conn.close()


<timed exec>:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: total: 41.1 s
Wall time: 42.7 s


### Show

In [20]:
%%time
df.head(100)

CPU times: total: 0 ns
Wall time: 0 ns


,bigAccountId,AmtFinanced,fltNetChgOff
0,370217,7361.26,6796.08
1,306072,22221.66,17702.55
2,245270,16712.35,4755.88
3,196070,15083.25,9161.37
4,239071,16493.79,15970.30
...,...,...,...
95,280455,16079.19,8385.76
96,280698,15401.43,311.46
97,260063,9111.85,-327.19
98,254047,11169.99,3548.62


#### Save as gzip parquet

In [11]:
str_filename = 'charge_offs.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path,
             compression='gzip')

#### Upload to S3

In [23]:
%%time

upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    str_local_path=str_local_path,
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}',
    str_bucket_name=str_project,
)

CPU times: total: 531 ms
Wall time: 1.76 s
